In [20]:
import yfinance as yf
import pandas as pd

# Show all columns without truncation
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Ask the user to input the ticker first
while True:
    ticker_symbol = input("Enter ticker symbol (e.g., 0700.HK): ").strip().upper()
    if ticker_symbol:
        break
    print("Ticker symbol cannot be empty. Please try again.")

# 1. Get daily data
ticker = yf.Ticker(ticker_symbol)
hist_daily = ticker.history(period="10y", interval="1d", auto_adjust=False)

if hist_daily.empty:
    raise SystemExit(f"No data found for ticker '{ticker_symbol}'. Please check the symbol and try again.")

# Convert index to datetime and remove timezone if present
hist_daily.index = pd.to_datetime(hist_daily.index)
if hist_daily.index.tz is not None:
    hist_daily.index = hist_daily.index.tz_localize(None)

# 2. Filter out future dates if any
today = pd.Timestamp.today().normalize()
hist_daily = hist_daily[hist_daily.index <= today]

# 3. Function to get the actual date and Close price
def get_actual_period_data(df, freq):
    grouped = df.groupby(pd.Grouper(freq=freq))

    result = grouped.agg(
        Actual_Date=('Close', lambda x: x.index[-1].strftime('%Y-%m-%d') if not x.empty else None),
        Close=('Close', 'last')
    )

    result = result.dropna(subset=['Actual_Date'])
    result = result.reset_index(drop=True)
    return result

# 4. Get quarter-end data
quarterly_data = get_actual_period_data(hist_daily, 'QE')

# =========================================================
# TRANSPOSE PROCESS (LEFT TO RIGHT BASED ON MOST RECENT DATE)
# =========================================================

# 1. Sort by Actual_Date from newest to oldest
quarterly_data = quarterly_data.sort_values(by='Actual_Date', ascending=False)

# 2. Set Actual_Date as index, then transpose
# so dates become columns from left to right
quarterly_data_transposed = quarterly_data.set_index('Actual_Date').T

# 3. Display result
print(f"\n--- {ticker_symbol} Quarter-End Closing Prices (Left to Right: Most Recent) ---")
quarterly_data_transposed

Enter ticker symbol (e.g., 0700.HK): bbca.jk

--- BBCA.JK Quarter-End Closing Prices (Left to Right: Most Recent) ---


Actual_Date,2026-09-18,2026-06-30,2026-03-31,2025-12-30,2025-09-30,2025-06-30,2025-03-27,2024-12-30,2024-09-30,2024-06-28,2024-03-28,2023-12-29,2023-09-29,2023-06-27,2023-03-31,2022-12-30,2022-09-30,2022-06-30,2022-03-31,2021-12-30,2021-09-30,2021-06-30,2021-03-31,2020-12-30,2020-09-30,2020-06-30,2020-03-31,2019-12-30,2019-09-30,2019-06-28,2019-03-29,2018-12-31,2018-09-28,2018-06-29,2018-03-30,2017-12-29,2017-09-29,2017-06-30,2017-03-31,2016-12-30,2016-09-30
Close,6300.0,5550.0,6450.0,8075.0,7625.0,8675.0,8500.0,9675.0,10325.0,9925.0,10075.0,9400.0,8825.0,9150.0,8750.0,8550.0,8550.0,7250.0,7975.0,7300.0,7000.0,6025.0,6215.0,6770.0,5420.0,5695.0,5525.0,6685.0,6070.0,5995.0,5510.0,5200.0,4830.0,4295.0,4660.0,4380.0,4060.0,3630.0,3310.0,3100.0,3140.0
